# Lesson 27: Imbalanced data: the fraud detection problem · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 27

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: How rare is fraud?

Store the number of fraud transactions in n_fraud and their share of all transactions (0 to 1) in share_fraud.

<details><summary>Hint</summary>

is_fraud is 1 or 0, so .sum() counts the frauds and .mean() gives their share.

</details>

In [ ]:
import pandas as pd

tx = pd.read_csv(DATA_URL.replace("loans.csv", "global/transactions.csv"))

n_fraud = ...
share_fraud = ...
print(n_fraud, share_fraud)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "import pandas as _pd\n_t = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/transactions.csv\"))\nassert n_fraud is not Ellipsis and int(n_fraud) == int(_t[\"is_fraud\"].sum()), \"n_fraud should count the rows where is_fraud is 1: tx[\\\"is_fraud\\\"].sum()\"\nassert share_fraud is not Ellipsis and abs(float(share_fraud) - _t[\"is_fraud\"].mean()) < 1e-9, \"share_fraud should be a share between 0 and 1: tx[\\\"is_fraud\\\"].mean()\"\n")

## Exercise 2: Balanced logistic regression

Split X and y with test_size=0.2, random_state=42 and stratify=y. Fit LogisticRegression(max_iter=1000, class_weight="balanced") on the training data, predict the test set, and store the recall in rec_b and the precision in prec_b.

<details><summary>Hint</summary>

train_test_split(X, y, test_size=0.2, random_state=42, stratify=y). LogisticRegression(max_iter=1000, class_weight="balanced").

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score

tx = pd.read_csv(DATA_URL.replace("loans.csv", "global/transactions.csv"))
tx["night"] = (tx["hour"] < 5).astype(int)
features = ["amount_usd", "is_online", "is_foreign", "new_merchant",
            "distance_from_home_km", "transactions_last_24h", "night"]
X = tx[features]
y = tx["is_fraud"]

X_train, X_test, y_train, y_test = ...
model = ...
model.fit(X_train, y_train)
preds = model.predict(X_test)

rec_b = ...
prec_b = ...
print(rec_b, prec_b)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "import pandas as _pd\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.linear_model import LogisticRegression as _LR\n_t = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/transactions.csv\"))\n_t[\"night\"] = (_t[\"hour\"] < 5).astype(int)\n_f = [\"amount_usd\", \"is_online\", \"is_foreign\", \"new_merchant\", \"distance_from_home_km\", \"transactions_last_24h\", \"night\"]\n_Xtr, _Xte, _ytr, _yte = _tts(_t[_f], _t[\"is_fraud\"], test_size=0.2, random_state=42, stratify=_t[\"is_fraud\"])\n_m = _LR(max_iter=1000, class_weight=\"balanced\").fit(_Xtr, _ytr)\nfrom sklearn.metrics import recall_score as _rs, precision_score as _ps\nassert list(X_test.index) == list(_Xte.index), \"Split with train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\"\n_p = _m.predict(_Xte)\nassert rec_b is not Ellipsis and abs(float(rec_b) - _rs(_yte, _p)) < 1e-9, \"rec_b should be recall_score(y_test, preds) for a model with class_weight=\\\"balanced\\\"\"\nassert prec_b is not Ellipsis and abs(float(prec_b) - _ps(_yte, _p)) < 1e-9, \"prec_b should be precision_score(y_test, preds)\"\n")

## Exercise 3: A higher cut-off

Using the balanced model provided, flag test transactions whose predicted probability of fraud is above 0.7. Store the number flagged in n_flagged and the number of real frauds among them in n_caught.

<details><summary>Hint</summary>

model.predict_proba(X_test)[:, 1] is the chance of fraud. Compare it with 0.7, then combine with (y_test == 1) using &.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score

tx = pd.read_csv(DATA_URL.replace("loans.csv", "global/transactions.csv"))
tx["night"] = (tx["hour"] < 5).astype(int)
features = ["amount_usd", "is_online", "is_foreign", "new_merchant",
            "distance_from_home_km", "transactions_last_24h", "night"]
X = tx[features]
y = tx["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)

p_fraud = ...
flag = ...
n_flagged = ...
n_caught = ...
print(n_flagged, n_caught)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "import pandas as _pd\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.linear_model import LogisticRegression as _LR\n_t = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/transactions.csv\"))\n_t[\"night\"] = (_t[\"hour\"] < 5).astype(int)\n_f = [\"amount_usd\", \"is_online\", \"is_foreign\", \"new_merchant\", \"distance_from_home_km\", \"transactions_last_24h\", \"night\"]\n_Xtr, _Xte, _ytr, _yte = _tts(_t[_f], _t[\"is_fraud\"], test_size=0.2, random_state=42, stratify=_t[\"is_fraud\"])\n_m = _LR(max_iter=1000, class_weight=\"balanced\").fit(_Xtr, _ytr)\n_fl = _m.predict_proba(_Xte)[:, 1] > 0.7\nassert n_flagged is not Ellipsis and int(n_flagged) == int(_fl.sum()), \"n_flagged should count probabilities above 0.7: (p_fraud > 0.7).sum()\"\nassert n_caught is not Ellipsis and int(n_caught) == int((_fl & (_yte.values == 1)).sum()), \"n_caught should count flagged transactions that are really fraud: (flag & (y_test == 1)).sum()\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=27) and take the quiz.